# Structered output

Models can be requested to provide their response in a format matching a given schema.This is useful for ensuring the output can be easily parsed and used in subsequent processing.Langchain supports multiples schema types and methods for enforcing structured output .

# Pydantic

pydantic models provide the richest features set with field validation , description and nested structures.

In [6]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

from langchain_groq import ChatGroq

model=ChatGroq(
    model="openai/gpt-oss-20b"
)

# model.invoke("started buring stomach , can i drink cold water ..if not what shall i do and whaat to avoid ?")

python-dotenv could not parse statement starting at line 1


In [3]:
os.environ["GROQ_API_KEY"]

'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'

In [7]:
model.invoke("why humans fart foul")

AIMessage(content='**Why do human farts smell so foul?**\n\n| What’s in a fart | Why it smells | How it’s produced |\n|------------------|----------------|-------------------|\n| **Major gases** – Nitrogen, oxygen, carbon‑dioxide, hydrogen, methane | *Odor‑free* – these are the bulk of the gas, but they don’t give the stink. | Produced when the digestive tract breaks down food and when you swallow air. |\n| **Trace odorants** – Sulfur‑containing compounds (hydrogen sulfide, methanethiol, dimethyl sulfide, etc.), short‑chain fatty acids, and some volatile organic compounds | These are the “smell” makers. Even tiny amounts (ppm‑level) are enough to be noticed. | Bacteria in the colon ferment undigested food. Different bacteria produce different by‑products. |\n| **Diet‑dependent compounds** – e.g., cooked onions (allicin → thiosulfinates), cruciferous veggies (glucosinolates → allyl isothiocyanate), garlic (allicin → diallyl disulfide), and certain proteins (amino‑acid breakdown → indole

In [2]:
from pydantic import BaseModel,Field
# baseclasss for creating pydantic models

class Movie(BaseModel):
    title:str=Field(description="title of the movie")
    year:int=Field(description="the year in which movie is released ")
    director:str=Field(description="the director of movide")
    rating:float=Field(description="the Movie rating our of 10")
    # rating:float=Field()

In [3]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x00000220B2FA8910>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000220B304BE10>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'title of the movie', 'type': 'string'}, 'year': {'description': 'the year in which movie is released ', 'type': 'integer'}, 'director': {'description': 'the director of movide', 'type': 'string'}, 'rating': {'description': 'the Movie rating o

In [4]:
model_with_structure.invoke("Provide me details about series - vikings valhala ")

Movie(title='Vikings: Valhalla', year=2022, director='Jonas Armstrong and Kim Friedman', rating=7.1)

### Message output alongide parsed structure

In [5]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    """ A Movie with Details. """
    title:str = Field(description=" Name of the movie")
    Year:int = Field(description=" The year is which movie is published")
    Director:str = Field(description=" Name of the director")
    rating:str = Field(description="rating out of 10")


model_with_structure=model.with_structured_output(Movie, include_raw=True)
response=model_with_structure.invoke("what the reason behind my sleep after having lumch ?")
response

{'raw': AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'crsdtwbmp', 'function': {'arguments': '{"Director":"Robert Stromberg","Year":2011,"rating":"6.0/10","title":"Sleeping Beauty"}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 37, 'prompt_tokens': 292, 'total_tokens': 329, 'completion_time': 0.055741152, 'completion_tokens_details': None, 'prompt_time': 0.020401064, 'prompt_tokens_details': None, 'queue_time': 0.050977656, 'total_time': 0.076142216}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e8c06-fa4e-7a32-8387-f3d6f39b233e-0', tool_calls=[{'name': 'Movie', 'args': {'Director': 'Robert Stromberg', 'Year': 2011, 'rating': '6.0/10', 'title': 'Sleeping Beauty'}, 'id': 'crsdtwbmp', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 292, 'ou

### Nested Structure

In [8]:
from pydantic import BaseModel,Field


class Actor(BaseModel):
    """ Actore in the cast"""
    name:str
    role:str=Field(description="Actor role in the movie")

class Movie(BaseModel):
    """ deatiled about a specific movie"""
    title:str
    year:str
    cast:list[Actor]
    genre:list[str]
    budget:float | None =Field(None,description="expense in movie")


model_with_structure = model.with_structured_output( Movie)
response=model_with_structure.invoke("Bhul bhulaiya")
response



Movie(title='Bhul Bhulaiyaa', year='2007', cast=[Actor(name='Akshay Kumar', role='Dr. Ranjit Singh')], genre=['Comedy', 'Horror', 'Romantic'], budget=None)

## TypedDict

TypedDict provides a simple alternative using python's built-in typing, ideal when you dont need runtime validation.

In [ ]:
from typing_extensions import TypedDict , Annotated

class MovieDict(TypedDict):
    """ Movie deatiled information """
    title:Annotated[str, ..., "thet title of the movie"]
    year: Annotated[str,...,"year in which movie is releasesd"]
    director:Annotated[str,...,"The director of the movie"]
    rating:Annotated[str,...,"raiting out of 10"]


model_with_structure=model.with_structured_output(MovieDict)
response=model_with_structure.invoke("Sex in the city")
response

{'director': 'Michael Patrick King',
 'rating': '7.3',
 'title': 'Sex and the City',
 'year': '2008'}

### Nested TypeDict

In [ ]:
from typing_extensions import TypedDict, Annotated

class actordict(TypedDict):
    """ detailed about  cast"""
    role : str
    name: str

class Moviedetails(TypedDict):
    """ detailed about movie """
    title:str
    year:int
    director:str
    cast:list[actordict]
    genre:list[str]
    # budget:Annotated[str,...,"total budget of the movie"]
    budget:float | None = Field(None,description="paisa kitna laga h pure pciture mei ")

model_with_structure=model.with_structured_output(Moviedetails)
response=model_with_structure.invoke("Bahubali movie ke baren mei bta re gandu")
response

{'budget': 180000000,
 'cast': [{'name': 'Prabhas', 'role': 'Shiva/Bahubali'},
  {'name': 'Rana Daggubati', 'role': 'Amarendra Baahubali'},
  {'name': 'Tamannaah', 'role': 'Devasena'},
  {'name': 'Anushka Shetty', 'role': 'Avantika'}],
 'director': 'S.S. Rajamouli',
 'genre': ['Action', 'Adventure', 'Drama'],
 'title': 'Bahubali',
 'year': 2015}

In [ ]:
model.profile 

{'max_input_tokens': 131072,
 'max_output_tokens': 8192,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': False,
 'tool_calling': True}

## Data Classes

A dataclass is a class typically containig mainy data , although there arent any restriction . you can create it using that @dataclass decorator 

In [ ]:
#creating a agent implementing structured output without with_structured_output using BaseModel

from pydantic import BaseModel,Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    name:str=Field(description="The name of the person")
    email:str=Field(description="email of the person")
    contactno:str=Field(description="phone no of the person")

agent=create_agent(
    model=model,
    response_format=ContactInfo         #auto selects Providerstrategy
)

result=agent.invoke(
    {
        "messages":
        [
            {
                "role":"user","The is Nathuram Godsen having gmail -GanduGandhi00@gmail.com phoneno- 6969226969, You're task is to extract the contact info"
                "content":
            }
        ]
    }
)

print(result,'\n')

{'messages': [HumanMessage(content="The is Nathuram Godsen having gmail -GanduGandhi00@gmail.com phoneno- 6969226969, You're task is to extract the contact info", additional_kwargs={}, response_metadata={}, id='71bd0062-0705-4e8c-aa7a-2b0caf4fe600'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '9mz5swaq7', 'function': {'arguments': '{"contactno":"6969226969","email":"GanduGandhi00@gmail.com","name":"Nathuram Godsen"}', 'name': 'ContactInfo'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 39, 'prompt_tokens': 292, 'total_tokens': 331, 'completion_time': 0.04691695, 'completion_tokens_details': None, 'prompt_time': 0.023782963, 'prompt_tokens_details': None, 'queue_time': 0.049463987, 'total_time': 0.070699913}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e8c0e-7704-7562-b7ce-ce5ebe4b6d

In [19]:
result

{'messages': [HumanMessage(content="The is Nathuram Godsen having gmail -GanduGandhi00@gmail.com phoneno- 6969226969, You're task is to extract the contact info", additional_kwargs={}, response_metadata={}, id='71bd0062-0705-4e8c-aa7a-2b0caf4fe600'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '9mz5swaq7', 'function': {'arguments': '{"contactno":"6969226969","email":"GanduGandhi00@gmail.com","name":"Nathuram Godsen"}', 'name': 'ContactInfo'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 39, 'prompt_tokens': 292, 'total_tokens': 331, 'completion_time': 0.04691695, 'completion_tokens_details': None, 'prompt_time': 0.023782963, 'prompt_tokens_details': None, 'queue_time': 0.049463987, 'total_time': 0.070699913}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e8c0e-7704-7562-b7ce-ce5ebe4b

In [16]:
result['structured_response']

ContactInfo(name='Nathuram Godsen', email='GanduGandhi00@gmail.com', contactno='6969226969')

In [ ]:
#implement Dataclass using TypedDict
from typing_extensions import TypedDict
from langchain.agents import create_agent

class contactinfo(TypedDict):   
    name:str
    phone:str
    email:str   #this tells the final output should dictnoray with exactly these fields

agent=create_agent(
    model=model,
    response_format=contactinfo  #Use structured output.
                                  #Instead of returning free-form text, extract information and populate the schema.
)

result=agent.invoke(
    {
        "messages":[
           {
            "role":"user",
            "content":"The is Nathuram Godsen having gmail -GanduGandhi00@gmail.com phoneno- 6969226969, You're task is to extract the contact info"
           } 
        ]
    }
)

result

{'messages': [HumanMessage(content="The is Nathuram Godsen having gmail -GanduGandhi00@gmail.com phoneno- 6969226969, You're task is to extract the contact info", additional_kwargs={}, response_metadata={}, id='1f210e64-6d82-42b9-933e-131627d3dce8'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'pkjcsvvaj', 'function': {'arguments': '{"email":"GanduGandhi00@gmail.com","name":"Nathuram Godsen","phone":"6969226969"}', 'name': 'contactinfo'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 40, 'prompt_tokens': 267, 'total_tokens': 307, 'completion_time': 0.049527821, 'completion_tokens_details': None, 'prompt_time': 0.020037978, 'prompt_tokens_details': None, 'queue_time': 0.051181551, 'total_time': 0.069565799}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e8c1f-6930-7411-87db-b3e82e29b47

In [23]:
result["structured_response"]

{'name': 'Nathuram Godsen',
 'phone': '6969226969',
 'email': 'GanduGandhi00@gmail.com'}

In [39]:
#Using dataclass
from dataclasses import dataclass
from langchain.agents import create_agent


@dataclass
class contactinfo:
    "contact ingormation for a person"
    name:str 
    phone:str
    email:str

agent=create_agent(
    model=model,
    response_format=contactinfo,
)

result3=agent.invoke({
    "messages":[
        {
            "role":"user",
            "content":"The is Nathuram Godsen having gmail -GanduGandhi00@gmail.com phoneno- 6969226969, You're task is to extract the contact info"

        }
    ]
})

agent
print(result3["structured_response"])


contactinfo(name='Nathuram Godsen', phone='6969226969', email='GanduGandhi00@gmail.com')
